# 13 IBM Quantum Hardware

## Theory
BB84 circuits can be executed on simulators or IBM Quantum backends. Hardware execution requires an IBM Quantum account and current backend access, so this notebook provides a simulator run plus optional runtime setup comments.

In [ ]:
# Common imports used in this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error, depolarizing_error, amplitude_damping_error

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# BB84 helper functions
def random_bits(n):
    """Generate n random classical bits."""
    return np.random.randint(0, 2, size=n)


def random_bases(n):
    """Generate n random bases: 0 means Z basis, 1 means X basis."""
    return np.random.randint(0, 2, size=n)


def prepare_bb84_qubit(bit, basis):
    """Create one BB84 qubit circuit.

    bit=0/1 is the secret bit.
    basis=0 prepares in Z basis; basis=1 prepares in X basis.
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc


def measure_bb84_qubit(qc, basis):
    """Measure the qubit in Bob's selected basis."""
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    return qc


def run_single_shot(qc, simulator=None, noise_model=None):
    """Run one circuit once and return measured bit as an integer."""
    simulator = simulator or AerSimulator(noise_model=noise_model)
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(max(counts, key=counts.get))


def sift_key(alice_bits, alice_bases, bob_bases, bob_bits):
    """Keep only positions where Alice and Bob used the same basis."""
    mask = alice_bases == bob_bases
    return alice_bits[mask], bob_bits[mask], mask


def qber(alice_key, bob_key):
    """Quantum Bit Error Rate: fraction of mismatched sifted bits."""
    if len(alice_key) == 0:
        return 0.0
    return float(np.mean(alice_key != bob_key))


def simulate_bb84(n=256, noise_model=None, eve=False):
    """Simulate BB84 with optional noise and optional intercept-resend Eve attack."""
    alice_bits = random_bits(n)
    alice_bases = random_bases(n)
    bob_bases = random_bases(n)
    bob_bits = []
    simulator = AerSimulator(noise_model=noise_model)

    for bit, alice_basis, bob_basis in zip(alice_bits, alice_bases, bob_bases):
        qc = prepare_bb84_qubit(int(bit), int(alice_basis))

        if eve:
            # Eve randomly measures and resends the qubit, which disturbs states
            # whenever she chooses the wrong basis.
            eve_basis = int(random_bases(1)[0])
            qc = measure_bb84_qubit(qc, eve_basis)
            eve_bit = run_single_shot(qc, simulator=simulator)
            qc = prepare_bb84_qubit(eve_bit, eve_basis)

        qc = measure_bb84_qubit(qc, int(bob_basis))
        bob_bits.append(run_single_shot(qc, simulator=simulator))

    bob_bits = np.array(bob_bits)
    alice_key, bob_key, mask = sift_key(alice_bits, alice_bases, bob_bases, bob_bits)
    return {
        "alice_bits": alice_bits,
        "alice_bases": alice_bases,
        "bob_bases": bob_bases,
        "bob_bits": bob_bits,
        "alice_key": alice_key,
        "bob_key": bob_key,
        "sift_mask": mask,
        "sifted_key_length": int(mask.sum()),
        "qber": qber(alice_key, bob_key),
    }

In [ ]:
# Build and run a small BB84 circuit batch on AerSimulator.
# To run on IBM hardware, install qiskit-ibm-runtime, save your token, and replace
# AerSimulator with a selected IBM backend from QiskitRuntimeService.
n = 16
alice_bits = random_bits(n)
alice_bases = random_bases(n)
bob_bases = random_bases(n)
circuits = []

for bit, alice_basis, bob_basis in zip(alice_bits, alice_bases, bob_bases):
    qc = prepare_bb84_qubit(int(bit), int(alice_basis))
    qc = measure_bb84_qubit(qc, int(bob_basis))
    circuits.append(qc)

simulator = AerSimulator()
compiled = transpile(circuits, simulator)
job = simulator.run(compiled, shots=1)
counts = job.result().get_counts()
bob_bits = np.array([int(max(c, key=c.get)) for c in counts])
alice_key, bob_key, mask = sift_key(alice_bits, alice_bases, bob_bases, bob_bits)

table = pd.DataFrame({
    "Metric": ["Backend used", "Total circuits", "Sifted key length", "QBER"],
    "Value": ["AerSimulator", n, int(mask.sum()), qber(alice_key, bob_key)],
})
display(table)

print("Optional hardware setup:")
print("from qiskit_ibm_runtime import QiskitRuntimeService")
print("service = QiskitRuntimeService(channel='ibm_quantum', token='YOUR_TOKEN')")
print("backend = service.least_busy(operational=True, simulator=False)")

## Conclusion
The notebook is ready for simulator execution and shows where IBM Quantum hardware access can be added. Hardware results may show real device noise and queue delays.